In [1]:
from irrigator.forecasts.arome_processing import (
    sync_static,
    sync_forecast,
    sync_dynamic,
    get_available_coverages,
    parse_coverage_id,
    find_available_run_dates,
    load_arome_daily_cache,
    ensure_dirs,
    DAILY_DIR,
    RAW_DIR,
)
from collections import defaultdict
import os

REPO_ROOT = os.path.dirname(os.getcwd())
os.chdir(REPO_ROOT)

print(os.getcwd())


results = sync_static(overwrite=False)

for d, status in sorted(results.items()):
    marker = "✓" if status == "cached" else ("↓" if status == "fetched" else "✗")
    print(f"  {marker} {d}: {status}")

results_dyn = sync_dynamic(overwrite=False)


results = sync_forecast(overwrite=False)

for d, status in sorted(results.items()):
    marker = "✓" if status == "cached" else ("↓" if status == "fetched" else "✗")
    print(f"  {marker} {d}: {status}")

    

2026-07-20 11:58:16,439 Syncing static archive...


/home/mbaldacchino/code/IrriGator


2026-07-20 12:01:08,127 [static] 2026-07-20: 98 calls
2026-07-20 12:01:13,683 [static] arome_daily_2026-07-20.nc → 23.2 MB
2026-07-20 12:01:13,685 Sync complete: 5 available, 4 cached, 1 fetched, 0 errors
2026-07-20 12:01:13,686 Syncing dynamic archive...


  ✓ 2026-07-16: cached
  ✓ 2026-07-17: cached
  ✓ 2026-07-18: cached
  ✓ 2026-07-19: cached
  ↓ 2026-07-20: fetched


2026-07-20 12:01:41,963 [dynamic] 2026-07-19T15: done
2026-07-20 12:02:08,839 [dynamic] 2026-07-19T18: done
2026-07-20 12:02:29,229 [dynamic] 2026-07-19T21: done
2026-07-20 12:02:50,396 [dynamic] 2026-07-20T00: done
2026-07-20 12:03:16,929 [dynamic] 2026-07-20T03: done
2026-07-20 12:03:17,090 FAIL TOTAL_PRECIPITATION__GROUND_OR_WATER_SURFACE___202 at 2026-07-20T09:00:00Z: 404
2026-07-20 12:03:20,982 [dynamic] arome_daily_2026-07-16.nc → 8 windows, 25.1 MB
2026-07-20 12:03:24,744 [dynamic] arome_daily_2026-07-17.nc → 8 windows, 24.8 MB
2026-07-20 12:03:28,549 [dynamic] arome_daily_2026-07-18.nc → 8 windows, 24.0 MB
2026-07-20 12:03:32,422 [dynamic] arome_daily_2026-07-19.nc → 8 windows, 23.7 MB
2026-07-20 12:03:33,941 [dynamic] arome_daily_2026-07-20.nc → 2 windows, 22.9 MB
2026-07-20 12:03:33,943 Syncing forecast archive...
2026-07-20 12:07:11,130 [forecast] 2026-07-18: 194 calls
2026-07-20 12:07:18,073 [forecast] arome_daily_2026-07-18.nc → 47.5 MB
2026-07-20 12:13:07,464 [forecast] 2

  ✓ 2026-07-16: cached
  ✓ 2026-07-17: cached
  ↓ 2026-07-18: fetched
  ↓ 2026-07-19: fetched
  ↓ 2026-07-20: fetched


# IFS

In [2]:
import os
import logging
from datetime import date, timedelta
from pathlib import Path
from irrigator.ingestion.ifs_ens_client import run_ifs_pipeline, load_ifs_daily

# Today only (default)
# daily_paths = run_ifs_pipeline()

# Or a date range:
daily_paths = run_ifs_pipeline(
    start=date(2026, 7, 6),
    end=date.today(),
    keep_raw=False,  # delete GRIBs after processing (default)
    #split_params=False
)

print(f"\nProcessed {len(daily_paths)} runs:")
for p in daily_paths:
    print(f"  {p.name}  ({p.stat().st_size / 1e6:.0f} MB)")


2026-07-20 12:18:43,780 IFS ENS 2026-07-06 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-07-06_00z.nc
2026-07-20 12:19:15,058 IFS ENS 2026-07-07 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-07-07_00z.nc
2026-07-20 12:19:46,579 IFS ENS 2026-07-08 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-07-08_00z.nc
2026-07-20 12:20:17,997 IFS ENS 2026-07-09 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-07-09_00z.nc
2026-07-20 12:20:49,429 IFS ENS 2026-07-10 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-07-10_00z.nc
2026-07-20 12:21:20,873 IFS ENS 2026-07-11 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-07-11_00z.nc
2026-07-20 12:21:52,347 IFS ENS 2026-07-12 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-07-12_00z.nc
2026-07-20 12:22:23,810 IFS ENS 2026-07-13 00Z already processed: data/processed/ifs_ens/ifs_daily_2026-07-13_00z.nc
2026-07-20 12:22:55,232 IFS ENS 2026-07-14 00Z already processed

<multiple>:   0%|          | 0.00/16.0G [00:00<?, ?B/s]

2026-07-20 12:42:45,566 Downloaded IFS ENS: data/raw/ifs_ens/ifs_ens_2026-07-20_00z.grib2 (17198.7 MB)
2026-07-20 12:42:45,567 [2026-07-20 00Z] Opening and slicing to bbox...


By downloading data from the ECMWF open data dataset, you agree to the terms: Attribution 4.0 International (CC BY 4.0). Please attribute ECMWF when downloading this data.


2026-07-20 12:44:05,677 [2026-07-20 00Z] Processing to daily...
2026-07-20 12:51:36,423 IFS ENS daily: 16 days, 50 members, 8 variables
2026-07-20 12:51:38,193 Saved IFS ENS daily: data/processed/ifs_ens/ifs_daily_2026-07-20_00z.nc (46.5 MB, 50 members, 16 days)
2026-07-20 12:51:40,667 [2026-07-20 00Z] Deleted raw GRIB (17199 MB freed)
2026-07-20 12:51:40,700 IFS pipeline complete: 15/15 runs processed



Processed 15 runs:
  ifs_daily_2026-07-06_00z.nc  (46 MB)
  ifs_daily_2026-07-07_00z.nc  (46 MB)
  ifs_daily_2026-07-08_00z.nc  (46 MB)
  ifs_daily_2026-07-09_00z.nc  (46 MB)
  ifs_daily_2026-07-10_00z.nc  (46 MB)
  ifs_daily_2026-07-11_00z.nc  (46 MB)
  ifs_daily_2026-07-12_00z.nc  (46 MB)
  ifs_daily_2026-07-13_00z.nc  (46 MB)
  ifs_daily_2026-07-14_00z.nc  (46 MB)
  ifs_daily_2026-07-15_00z.nc  (47 MB)
  ifs_daily_2026-07-16_00z.nc  (47 MB)
  ifs_daily_2026-07-17_00z.nc  (46 MB)
  ifs_daily_2026-07-18_00z.nc  (46 MB)
  ifs_daily_2026-07-19_00z.nc  (46 MB)
  ifs_daily_2026-07-20_00z.nc  (46 MB)


In [5]:
import xarray as xr
import rioxarray as rxr
import geopandas as gpd

edouard = gpd.read_file(
    "/home/mbaldacchino/code/IrriGator/configs/IRRIGATOR PARCELLES/boundaries/boundaries.shp"
)
print(edouard)

          CLIENT_NAM          FARM_NAME       FIELD_NAME  POLYGONTYP  \
0   SEP DES POUYADES   EARL CHEZ BILLAC           BILLAC           0   
1   SEP DES POUYADES  SCEA DES POUYADES       CLAUZUROUX           0   
2   SEP DES POUYADES   EARL CHEZ BILLAC    FAURES MARAIS           0   
3   SEP DES POUYADES  SCEA DES POUYADES        FEUILLADE           0   
4   SEP DES POUYADES  SCEA DES POUYADES  GRENOUILLET 0,6           0   
5   SEP DES POUYADES  SCEA DES POUYADES  GRENOUILLET 2,4           0   
6   SEP DES POUYADES  SCEA DES POUYADES            GUIDE           0   
7   SEP DES POUYADES  SCEA DES POUYADES           LUSSAC           0   
8   SEP DES POUYADES  SCEA DES POUYADES         MARAIS 1           0   
9   SEP DES POUYADES  SCEA DES POUYADES         MARAIS 2           0   
10  SEP DES POUYADES  SCEA DES POUYADES            MILHE           0   
11  SEP DES POUYADES  SCEA DES POUYADES    POUYADES NORD           0   
12  SEP DES POUYADES     SCEA DE POUZET    SOULET MARAIS        

In [7]:
edouard["centroid"] = edouard.centroid

/tmp/ipykernel_1915/2610382648.py:1: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  edouard["centroid"] = edouard.centroid


In [8]:
edouard

,CLIENT_NAM,FARM_NAME,FIELD_NAME,POLYGONTYP,CLIENT_ID,FARM_ID,FIELD_ID,ORG_ID,geometry,centroid
0,SEP DES POUYADES,EARL CHEZ BILLAC,BILLAC,0,4c8bf6bc-eded-4337-b490-07db4392c74b,642d28a5-0000-1000-6143-e1e1e13aeb58,c405a2cb-92c9-4439-9527-b81d9be7c0c7,489713,"POLYGON ((0.41104 45.38448, 0.41209 45.38647, ...",POINT (0.41508 45.38383)
1,SEP DES POUYADES,SCEA DES POUYADES,CLAUZUROUX,0,4c8bf6bc-eded-4337-b490-07db4392c74b,60002b2f-e5d3-49d0-b569-92070dbb7c2e,b6de8403-ecad-4102-809d-5acbb04d29fc,489713,"POLYGON ((0.35866 45.41049, 0.35748 45.41044, ...",POINT (0.3595 45.4109)
2,SEP DES POUYADES,EARL CHEZ BILLAC,FAURES MARAIS,0,4c8bf6bc-eded-4337-b490-07db4392c74b,642d28a5-0000-1000-6143-e1e1e13aeb58,642d6e13-0000-1000-5216-e1e1e13aeb58,489713,"POLYGON ((0.4032 45.39582, 0.40308 45.39578, 0...",POINT (0.40154 45.39656)
3,SEP DES POUYADES,SCEA DES POUYADES,FEUILLADE,0,4c8bf6bc-eded-4337-b490-07db4392c74b,60002b2f-e5d3-49d0-b569-92070dbb7c2e,b4a860f4-5e26-46e9-aa2e-50e182d9d68f,489713,"POLYGON ((0.39938 45.3817, 0.40006 45.3818, 0....",POINT (0.40337 45.38072)
4,SEP DES POUYADES,SCEA DES POUYADES,"GRENOUILLET 0,6",0,4c8bf6bc-eded-4337-b490-07db4392c74b,60002b2f-e5d3-49d0-b569-92070dbb7c2e,745d84b7-4a1e-4bcb-a4f3-c254106f8c1c,489713,"POLYGON ((0.39035 45.40254, 0.39036 45.40256, ...",POINT (0.3905 45.40192)
5,SEP DES POUYADES,SCEA DES POUYADES,"GRENOUILLET 2,4",0,4c8bf6bc-eded-4337-b490-07db4392c74b,60002b2f-e5d3-49d0-b569-92070dbb7c2e,11af66c9-7a07-4a98-b3c3-660726da22b0,489713,"POLYGON ((0.38674 45.40442, 0.38704 45.40221, ...",POINT (0.38623 45.40338)
6,SEP DES POUYADES,SCEA DES POUYADES,GUIDE,0,4c8bf6bc-eded-4337-b490-07db4392c74b,60002b2f-e5d3-49d0-b569-92070dbb7c2e,7e1ad678-9c3c-4fdb-9f9d-b6d1f396b71c,489713,"POLYGON ((0.39503 45.3745, 0.39545 45.37456, 0...",POINT (0.39938 45.37163)
7,SEP DES POUYADES,SCEA DES POUYADES,LUSSAC,0,4c8bf6bc-eded-4337-b490-07db4392c74b,60002b2f-e5d3-49d0-b569-92070dbb7c2e,44ea0d37-e0f7-4a5e-8be1-4f3461bb43d7,489713,"POLYGON ((0.36313 45.33063, 0.3632 45.33065, 0...",POINT (0.36038 45.32928)
8,SEP DES POUYADES,SCEA DES POUYADES,MARAIS 1,0,4c8bf6bc-eded-4337-b490-07db4392c74b,60002b2f-e5d3-49d0-b569-92070dbb7c2e,2f13049e-5b5c-422c-94b0-c9910fd94f1b,489713,"POLYGON ((0.37243 45.40601, 0.37231 45.4067, 0...",POINT (0.37349 45.40656)
9,SEP DES POUYADES,SCEA DES POUYADES,MARAIS 2,0,4c8bf6bc-eded-4337-b490-07db4392c74b,60002b2f-e5d3-49d0-b569-92070dbb7c2e,e49a3c8e-9d62-4aaf-bbb5-4c0a0cfbe477,489713,"POLYGON ((0.37673 45.4055, 0.37565 45.404, 0.3...",POINT (0.37562 45.40472)
